# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [ ]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder

import polars as pl

In [ ]:
# If you have a specific config file, load it here:
# settings.load_from_yaml("data/MultiplEYE_<...>/multipleye_settings_preprocessing.yaml")

In [ ]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [ ]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Recalculate:     {settings.RECALCULATE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [ ]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [ ]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

In [ ]:
multipleye.prepare_session_level_information()

In [ ]:
multipleye.skipped_sessions.keys()

In [ ]:
sid = '007_SV_CH_1_ET1'

In [ ]:
sess = multipleye.skipped_sessions[sid]

In [ ]:
multipleye.skipped_sessions[sid].logfile = multipleye._load_session_logfile(sid)

In [ ]:
(multipleye.skipped_sessions[sid].completed_stimuli_ids,
multipleye.skipped_sessions[sid].completed_stimuli_names,
multipleye.skipped_sessions[sid].stimuli_trial_mapping,
) = multipleye._load_session_completed_stimuli(sid)

In [ ]:
parsed_answers = preprocessing.parse_answers_from_logfile(sess.logfile, sess.stimuli_trial_mapping)

In [ ]:
question_order_csv = (
                    sess.session_folder_path
                    / "logfiles"
                    / "question_order_versions.csv"
                )

In [ ]:
answers_csv = sess.sid.answers_dir / f"{sess.sid}_answers.csv"

In [ ]:
source = "logfile"

In [ ]:
sess.stimuli_trial_mapping

In [ ]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

In [ ]:
from pathlib import Path

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [ ]:
preprocessing.run_preflight_check(multipleye)

## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [ ]:
multipleye.convert_edf_to_asc()

Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [ ]:
multipleye.prepare_session_level_information()

In [ ]:
# print an overview on the data collection and the sessions
multipleye

## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [ ]:
for sess in multipleye.skipped_session_ids:
    print(sess)

## Stage 4: Comprehension Question Answers
In addition to gaze data, each session contains answers to comprehension questions.
These are extracted from the ASC messages. The answers are matched to the stimulus order using the `question_order_versions.csv` file in the session's logfiles folder.

In [ ]:
answers_csv = sid.answers_dir / f"{sid}_answers.csv"
question_order_csv = (
    sess.session_folder_path / "logfiles" / "question_order_versions.csv"
)


In [ ]:
answers_csv

In [ ]:
question_order_csv

In [ ]:
sess.stimuli_trial_mapping

In [ ]:
sess = multipleye.skipped_sessions[sid]

In [ ]:
parsed_answers = preprocessing.parse_answers_from_logfile(
                            sess.logfile, sess.stimuli_trial_mapping
                        )
source = "logfile"

In [ ]:
sess.sid

In [ ]:
parsed_answers

In [ ]:
for session_identifier in multipleye.skipped_sessions.keys():
    print(session_identifier)

In [ ]:
preprocessing.collect_session_answers(
    question_order_csv=question_order_csv,
    stimuli_trial_map=sess.stimuli_trial_mapping,
    stimuli=sess.stimuli,
    parsed_answers=parsed_answers,
    out_path=answers_csv,
    source=source,
    completed_stimuli_ids=sess.completed_stimuli_ids,
)

In [ ]:
gaze = preprocessing.load_trial_level_raw_data(
                sess.sid,
                trial_columns=settings.TRIAL_COLS,
                load_metadata=True,
            )

In [ ]:
parsed_answers = preprocessing.parse_answers_from_messages(
                            gaze.messages
                        )
source = "asc"

In [ ]:
parsed_answers